In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root added to Python path:")
print(PROJECT_ROOT)

✅ Project root added to Python path:
c:\GitHub Projects\VWAP-probability-band-engine


In [ ]:
import MetaTrader5 as mt5

from src.config import CONFIG
from src.loaders import load_mt5_live, fetch_mt5_history
from src.signals import SignalResult, generate_signal, regime_gate, apply_filters
from src.live_runner import run_live
from src.startup import prepare_live_startup
from src.exports import save_artifacts
from src.mql5_overlay import write_mql5_overlay
from src.engine import EngineState, update_engine_state

### Section 13 — Live Mode (MT5 Adapter)

Live mode uses the same core probability band engine as the backtest workflow. The main difference is the data source — MT5 live bars instead of historical CSV data.

This section defines the MT5-connected live runner and prepares the artifacts required for live monitoring, signal updates, and external overlays.

The live notebook is designed to reuse the shared `src/` modules wherever possible, while keeping MT5-specific orchestration here.

In [3]:
print("✅ Live mode functions loaded")
print(f"   Instrument : {CONFIG['instrument']}")
print(f"   Timeframe  : {CONFIG['timeframe']}")
print("   run_live() available")
print("   save_artifacts() available")
print("   fetch_mt5_history() available")

✅ Live mode functions loaded
   Instrument : US100.cash
   Timeframe  : M1
   run_live() available
   save_artifacts() available
   fetch_mt5_history() available


In [4]:
# Optional MT5 historical backfill for local reuse
# import MetaTrader5 as mt5
#
# df_hist = fetch_mt5_history(
#     symbol=CONFIG['instrument'],
#     timeframe_mt5=mt5.TIMEFRAME_M1,
#     lookback_days=30,
#     save_csv=True,
#     csv_dir="data",
# )
#
# df_hist.head(3)

In [5]:
from pathlib import Path
import pandas as pd

artifacts_dir = PROJECT_ROOT / "artifacts" / "tables"

prob_table = pd.read_parquet(artifacts_dir / "prob_table_marginal.parquet")
prob_table_trend = pd.read_parquet(artifacts_dir / "prob_table_trend.parquet")

print("✅ Live probability tables loaded")
print(f"   Marginal rows : {len(prob_table):,}")
print(f"   Trend rows    : {len(prob_table_trend):,}")

✅ Live probability tables loaded
   Marginal rows : 21
   Trend rows    : 42


---
### Section 17 — MQL5 Overlay Generator

This section writes the MQL5 indicator source file to disk.
The indicator reads `live_artifacts/live_state.json` once per bar
and draws VWAP, sigma bands, zone label, and signal alert on the chart.

It contains no calculation logic — it is purely a rendering layer
that consumes the Python engine's JSON output.

**To install:**
1. Copy the generated `.mq5` file to your MT5 `Indicators/` folder
2. Compile in MetaEditor (F5)
3. Attach to the chart — set the JSON path to match your Python output path

In [6]:
overlay_path = write_mql5_overlay(
    output_dir=str(PROJECT_ROOT / "live_artifacts" / "exports"),
    filename="VWAP_Overlay.mq5"
)

print("✅ Overlay saved to:")
print(overlay_path)

✅ MQL5 source written → c:\GitHub Projects\VWAP-probability-band-engine\live_artifacts\exports\VWAP_Overlay.mq5
✅ Copied to MT5 Indicators → C:\Users\Michael\AppData\Roaming\MetaQuotes\Terminal\49CDDEAA95A409ED22BD2287BB67CB9C\MQL5\Indicators\VWAP_Overlay.mq5
   File size: 22,283 bytes

Next steps:
  1. In MT5 → Navigator → Indicators → right-click → Refresh
  2. Double-click VWAP_Overlay to open in MetaEditor
  3. Press F7 to compile — should show 0 errors
  4. Compile and attach to your chart
  5. Run the live engine in the notebook
✅ Overlay saved to:
None


In [ ]:
#### LIVE STARTUP SETTINGS

#### "now"    = current behaviour: normal recent MT5 warmup
#### "preset" = rebuild from a named session open
#### "manual" = rebuild from a specific UK time
#### Safe default is "now", so the model behaves like it did before.

SESSION_START_MODE = "now"
#### Options: "now", "preset", "manual"

SESSION_PRESET = "ny_open"
#### Options: "london_open", "ny_open", "asia_open"

ANCHOR_OFFSET_MINUTES = -30
#### Used only for preset mode:
#### -30 = start 30 minutes before selected session open
#### 0   = start exactly at selected session open
#### 30  = start 30 minutes after selected session open

MANUAL_START_UK = "16:00"
#### Used only for manual mode.
#### This is UK time, not FTMO/server time.

BROKER_TIMEZONE_NAME = "Europe/Athens"
#### Display only.
#### For FTMO-style GMT+2/GMT+3 server display, use "Europe/Athens".
#### For no broker/server display, set this to None.

### Section 18 — Run Live Engine

In [ ]:
# LIVE MODE — run this cell last, after all cells above are done
# MT5 must be open and logged in before running this

CONFIG['instrument'] = 'US100.cash'

if 'prob_table' not in dir():
    print("❌ prob_table not found — load/export the probability table first")
    raise SystemExit

mt5.shutdown()
if not mt5.initialize():
    print(f"❌ MT5 initialize() failed — error: {mt5.last_error()}")
    print("   Make sure MT5 is open, logged in, and Algo Trading is enabled")
    print("   Tools → Options → Expert Advisors → tick 'Allow algorithmic trading'")
    raise SystemExit

print(f"✅ MT5 connected — {mt5.terminal_info().name}")
print(f"   Account : {mt5.account_info().login} | Server: {mt5.account_info().server}")

symbol_info = mt5.symbol_info(CONFIG['instrument'])

if symbol_info is None:
    print(f"\n❌ Symbol '{CONFIG['instrument']}' not found on this account.")
    print("   Symbols on this broker containing US/NAS/TECH:")
    all_syms = mt5.symbols_get()
    matches = [s.name for s in all_syms
               if any(x in s.name.upper() for x in ['US1', 'NAS', 'TECH', 'NDX', 'USTEC'])]
    print(f"   {matches[:20]}")
    print("\n   Update CONFIG['instrument'] to the correct name above, then re-run.")
    mt5.shutdown()
    raise SystemExit

live_spread = symbol_info.ask - symbol_info.bid
print(f"\n✅ Symbol found: {symbol_info.name}")
print(f"   Bid: {symbol_info.bid}  Ask: {symbol_info.ask}")
print(f"   Live spread right now: {live_spread:.1f} points")
print(f"   ↳ If this looks right, set CONFIG['spread_cost'] = {live_spread:.1f} and re-run relevant calibration cells")
print(f"   ↳ (only matters for outcome labelling accuracy, not for live signal generation)")

initial_df, session_info = prepare_live_startup(
    symbol=CONFIG["instrument"],
    timeframe_mt5=mt5.TIMEFRAME_M1,
    session_start_mode=SESSION_START_MODE,
    session_preset=SESSION_PRESET,
    anchor_offset_minutes=ANCHOR_OFFSET_MINUTES,
    manual_start_uk=MANUAL_START_UK,
    broker_timezone_name=BROKER_TIMEZONE_NAME,
)

print("\n🟢 Starting live engine — press the ■ stop button in Jupyter to stop\n")
try:
    run_live(
        symbol=CONFIG['instrument'],
        timeframe_mt5=mt5.TIMEFRAME_M1,
        config=CONFIG,
        prob_table=prob_table,
        marginal_table=prob_table,
        load_mt5_live=load_mt5_live,
        EngineState=EngineState,
        update_engine_state=update_engine_state,
        generate_signal=generate_signal,
        regime_gate=regime_gate,
        apply_filters=apply_filters,
        initial_df=initial_df,
        session_info=session_info,
    )
finally:
    mt5.shutdown()
    print("\n🔴 MT5 connection closed cleanly")

✅ MT5 connected — FTMO MetaTrader 5
   Account : 511240067 | Server: FTMO-Server

✅ Symbol found: US100.cash
   Bid: 28949.15  Ask: 28951.05
   Live spread right now: 1.9 points
   ↳ If this looks right, set CONFIG['spread_cost'] = 1.9 and re-run relevant calibration cells
   ↳ (only matters for outcome labelling accuracy, not for live signal generation)

🟢 Starting live engine — press the ■ stop button in Jupyter to stop

✅ MT5 Live loaded: 200 bars for US100.cash
✅ Live engine warmed up on 200 bars
🟢 Live mode active — US100.cash | Ctrl+C to stop

🔴 Live mode stopped

🔴 MT5 connection closed cleanly
